In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [24]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate

In [4]:
loader = PyPDFLoader("../data/AIML.pdf") 
docs = loader.load()
len(docs)

22

In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
splitted_data = splitter.split_documents(docs)
len(splitted_data)

37

In [6]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [9]:
vector_store = Chroma.from_documents(
    documents = splitted_data,
    embedding = embeddings
)

In [10]:
query = "Machine learning and AI Engineer Content"
data = vector_store.similarity_search(query=query)

In [15]:
context = ""
for doc in data:
    context += doc.page_content +"\n"



In [17]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-5.6-luna")

### Chain - Context_generate | Prompt | LLM | Stranger

In [45]:
def get_context(query:str):
    data = vector_store.similarity_search(query=query)
    context = "" 
    for doc in data:
        context += doc.page_content +"\n"

    return {
        "context":context,
        "question":query
    }
    

In [46]:
prompt = PromptTemplate.from_template("""
    You are a helpful assistent and provide answerd based on the context for user question.
    if  you don't know the answer, then you can say that 'I don't know.' 
    Context: {context}
    Question: {question}
    """)

In [47]:
rag_chain = get_context | prompt | llm

In [53]:
res = rag_chain.invoke("What is the content java DSA course?")

In [54]:
print(res.content)

The provided curriculum does not include a Java DSA course. It focuses on:

- AI landscape and career paths  
- Python programming  
- SQL and data warehousing  
- Pandas for data handling  
- Machine learning and deep learning  
- NLP and generative AI  
- Agentic AI systems  
- Deployment, APIs, and MLOps  

The content of a Java Data Structures and Algorithms course is not specified in the provided information.
